# 12 · Fairness diagnostics and mitigation

**Question.** Do the poverty and VUP coverage-gap models work equally well for different kinds of households? For the poverty model, which mitigation narrows the gaps without losing accuracy?

| | |
|---|---|
| **Data** | Development set only (12,032 households; 2,457 high-need households for the VUP model). Out-of-fold predictions from the same 5 cluster-based folds. **The holdout is not used**; it is kept for notebook 14. |
| **Groups** | Member with a disability, head age group, province, urban/rural, head's sex, household size band, main income source |
| **Metrics by group** | ROC-AUC and PR-AUC (groups with at least 20 positives); recall (people) and precision at a **single national cut-off** (top 20% of households by predicted score); selection rate; calibration (mean predicted vs observed); mean prediction error (predicted minus observed); Brier score |
| **Intervals** | 95% cluster bootstrap, 200 draws, clusters resampled within strata, cut-off recomputed in each draw |
| **Code** | `src/models/fairness.py` |
| **Outputs** | `reports/fairness/*.csv`, `reports/figures/fairness/*.png`, `reports/fairness/mitigation_choice.json` |

**Mitigations tested on the poverty model (LightGBM, default settings):**
1. **Interaction features:** explicit disability, head-age and urban indicators, and their interactions with household size, dependants, food share, assets and education.
2. **Re-weighting:**
   - (a) *subgroup-balanced* weights, which up-weight small disability × head-age × area cells;
   - (b) *class-balanced* weights.
3. **Group-aware calibration:**
   - (a) Platt scaling with intercepts for disability, head-age, province and area, cross-fitted across folds;
   - (b) **policy option:** separate urban and rural cut-offs, with the national budget split by each area's predicted number of poor people.

**Selection rule (fixed in advance, `fairness.choose_approach`).**
1. Keep approaches whose mean fold PR-AUC is within one standard error of the best.
2. Among them, choose the best average rank on recall gap, worst-group recall and calibration gap.
3. Ties go to the simpler approach.

The unmitigated model is a candidate. The separate cut-offs are a selection policy, not a model, so they are assessed on top of the chosen model.

In [ ]:
import json
import sys
import warnings
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "models" / "common.py").exists())
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from IPython.display import Image, display
from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline

from src.models import common as cm
from src.models import fairness as fr
from src.models import plots as pl
from src.models import poverty_model as pm
from src.models import training as tr
from src.models import vup_gap_model as vm

pd.set_option("display.width", 230)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.3f}".format)
OUT = ROOT / "reports" / "fairness"
FIG = ROOT / "reports" / "figures" / "fairness"
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

df = cm.load_frame().merge(cm.load_splits(), on=["hhid", "clust"])
dev = fr.add_groups(df[df["split"].eq("development")].reset_index(drop=True))
spec = pm.FEATURE_SETS["full"]
folds = dev["fold"]
hn = vm.load_population()
hdev = fr.add_groups(hn[hn["split"].eq("development")].reset_index(drop=True))
print(f"Development: {len(dev):,} households; high-need development: {len(hdev):,} ({int(hdev['vup_any'].sum())} VUP recipients)")
display(pd.concat({d: dev.groupby(d).size() for d in fr.DIMENSIONS}).rename("households").to_frame().T)

## 1. Baseline diagnostics: both models

Default models from notebooks 09 and 10 (LightGBM for poverty, logistic regression for VUP coverage), out-of-fold on the development set.

In [ ]:
pipes, cols = tr.make_pipelines(spec, ["lightgbm"])
dev["p_base"] = fr.cv_predict(pipes["lightgbm"], dev, cols["lightgbm"], "poor", folds)
vpipes, vcols = tr.make_pipelines(vm.FEATURES, ["logistic_regression"])
hdev["v_base"] = fr.cv_predict(vpipes["logistic_regression"], hdev, vcols["logistic_regression"], "vup_any", hdev["fold"])
print("Poverty, overall:", {k: round(v, 3) for k, v in cm.evaluate(dev["poor"], dev["p_base"], dev["weight"], dev["pop_wt"]).items()
                            if k in ("roc_auc", "pr_auc", "recall_people_top20", "precision_top20", "calibration_slope")})
print("VUP coverage, overall:", {k: round(v, 3) for k, v in cm.evaluate(hdev["vup_any"], hdev["v_base"], hdev["weight"], hdev["pop_wt"]).items()
                                 if k in ("roc_auc", "pr_auc", "recall_top20", "precision_top20", "calibration_slope")})
pov_tab = fr.group_table_ci(dev, "p_base", "poor", B=200)
vup_tab = fr.group_table_ci(hdev, "v_base", "vup_any", B=200)
pov_tab.to_csv(OUT / "group_metrics_poverty_baseline.csv", index=False)
vup_tab.to_csv(OUT / "group_metrics_vup_baseline.csv", index=False)
show = ["dimension", "group", "households", "positives", "observed_rate", "mean_predicted", "mean_error", "mean_error_ci_low", "mean_error_ci_high",
        "roc_auc", "pr_auc", "selection_rate", "precision_top20", "recall_people_top20", "recall_people_top20_ci_low", "recall_people_top20_ci_high"]
pov_tab[show]

In [ ]:
def fairness_chart(tab, title, subtitle, path, recall_label):
    t = tab.copy()
    t.index = [f"{d.replace('_', ' ')}: {g}" for d, g in zip(t["dimension"], t["group"])]
    n = len(t)
    h = 1.9 + 0.27 * n
    fig, axes = pl.figure(title, subtitle, width=14, height=h, ncols=3)
    specs = [("recall_people_top20", recall_label, 100, "{:.0f}", None),
             ("mean_error", "Mean prediction error, pp (predicted - observed)", 100, "{:+.1f}", 0.0),
             ("roc_auc", "ROC-AUC within group", 1, "{:.2f}", None)]
    for i, (ax, (col, label, scale, fmt, ref)) in enumerate(zip(axes, specs)):
        tab_ = pd.DataFrame({"mean": t[col], "ci_low": t[f"{col}_ci_low"], "ci_high": t[f"{col}_ci_high"], "households": t["households"]})
        tab_ = tab_.fillna({"ci_low": tab_["mean"], "ci_high": tab_["mean"]})
        if tab_["mean"].isna().all():
            continue
        pl.dot_ci(ax, tab_.fillna(np.nan), scale, pl.SERIES[[2, 1, 0][i]], label_fmt=fmt, reference=ref, show_n=(i == 0))
        if i:
            ax.set_yticks(ax.get_yticks(), [""] * n)
        ax.set_title(label, fontsize=9.5, color=pl.INK["secondary"], loc="left")
    return pl.save(fig, path, top=1 - 1.3 / h, bottom=0.5 / h, left=0.22, wspace=0.12)

overall_pov = cm.evaluate(dev["poor"], dev["p_base"], dev["weight"], dev["pop_wt"])["recall_people_top20"]
p = fairness_chart(pov_tab, "Poverty model by group (development, out-of-fold)",
                   f"National top-20% cut-off. Recall = share of the group's poor people selected (overall {100 * overall_pov:.0f}%). 95% cluster-bootstrap intervals; AUC only where >= 20 poor households",
                   FIG / "baseline_poverty_by_group.png", "Poor people selected, %")
display(Image(str(p)))

In [ ]:
vup_tab[show]

In [ ]:
p = fairness_chart(vup_tab.rename(columns={}), "VUP coverage model by group (high-need households, development, out-of-fold)",
                   "Top 20% of high-need households by predicted coverage. Recall = share of the group's VUP recipients (people) in that top 20%. AUC only where >= 20 recipients",
                   FIG / "baseline_vup_by_group.png", "VUP recipients in top 20%, %")
display(Image(str(p)))
base_summary = {"poverty": fr.fairness_summary(pov_tab), "vup_coverage": fr.fairness_summary(vup_tab)}
pd.DataFrame(base_summary)

**Reading the baseline.** At a single national cut-off, groups with lower poverty rates (urban, younger heads, non-farm households) have a smaller share of their poor selected. That is partly a property of any national cut-off, not only of the model. The calibration gaps show where the model systematically under- or over-predicts. The mitigations below target both.

## 2. Poverty model: mitigations

In [ ]:
def lgbm(**kw):
    return LGBMClassifier(random_state=cm.RANDOM_STATE, verbose=-1, **kw)

spec_int = cm.FeatureSpec("full_plus_interactions", list(cm.GROUPS), extra=fr.INTERACTION_FEATURES)
cm.assert_no_leakage(spec_int, "poverty")
pipe_int = Pipeline([("groups", fr.GroupFeatureAdder()),
                     ("prep", cm.make_preprocessor(spec_int, scale=False).set_output(transform="pandas")),
                     ("model", lgbm())])
pipe_cls = Pipeline([("prep", cm.make_preprocessor(spec, scale=False).set_output(transform="pandas")), ("model", lgbm(class_weight="balanced"))])

scores = {"baseline": dev["p_base"]}
scores["1_interaction_features"] = fr.cv_predict(pipe_int, dev, spec.columns, "poor", folds)
scores["2a_subgroup_balanced_weights"] = fr.cv_predict(pipes["lightgbm"], dev, cols["lightgbm"], "poor", folds,
                                                       weight_multiplier=fr.subgroup_balanced_multiplier(dev))
scores["2b_class_balanced_weights"] = fr.cv_predict(pipe_cls, dev, spec.columns, "poor", folds)
scores["3a_group_recalibration"] = fr.cross_fit_calibration(dev["p_base"], dev, "poor", folds)
COMPLEXITY = {"baseline": 0, "1_interaction_features": 1, "2a_subgroup_balanced_weights": 1, "2b_class_balanced_weights": 1, "3a_group_recalibration": 2}
print("Weight multipliers (subgroup-balanced), by cell:")
mult = fr.subgroup_balanced_multiplier(dev)
display(mult.groupby(dev[list(fr.BALANCE_CELLS)].astype(str).agg(" | ".join, axis=1)).first().sort_values().to_frame("multiplier"))

In [ ]:
rows, tables = [], {}
for name, s in scores.items():
    d = dev.assign(score=s.to_numpy())
    fs = fr.fold_scores(d, d["score"], "poor", folds, "pr_auc")
    e = cm.evaluate(d["poor"], d["score"], d["weight"], d["pop_wt"])
    t = fr.group_table(d, "score", "poor")
    tables[name] = t
    rows.append({"approach": name, "pr_auc_mean": fs.mean(), "pr_auc_se": fs.std() / np.sqrt(len(fs)), "roc_auc": e["roc_auc"],
                 "recall_people_top20": e["recall_people_top20"], "precision_top20": e["precision_top20"], "brier": e["brier"],
                 "calibration_slope": e["calibration_slope"], "calibration_in_the_large": e["calibration_in_the_large"],
                 **fr.fairness_summary(t), "complexity": COMPLEXITY[name]})
res = pd.DataFrame(rows).set_index("approach")
best, ranked = fr.choose_approach(res)
display(ranked[["pr_auc_mean", "pr_auc_se", "eligible", "roc_auc", "recall_people_top20", "calibration_slope", "calibration_in_the_large",
                "recall_gap", "worst_group_recall", "mean_abs_calibration_gap", "max_abs_calibration_gap", "auc_gap", "fairness_rank", "complexity"]])
print(f"Chosen approach (rule: PR-AUC within 1 SE of best, then best fairness rank, then simplest): {best}")

In [ ]:
# Policy layer: separate urban/rural cut-offs on top of the chosen score
d_best = dev.assign(score=scores[best].to_numpy())
inc_quota = fr.area_quota_inclusion(d_best["score"], d_best["weight"], d_best["pop_wt"], d_best["urban"])
t_quota = fr.group_table(d_best, "score", "poor", inclusion=inc_quota)
y, w_, pw_ = d_best["poor"].to_numpy(), d_best["weight"].to_numpy(), d_best["pop_wt"].to_numpy()
quota_overall = {"recall_people_top20": (inc_quota * pw_ * y).sum() / (pw_ * y).sum(), "precision_top20": (inc_quota * w_ * y).sum() / (inc_quota * w_).sum()}
nat_overall = cm.evaluate(d_best["poor"], d_best["score"], d_best["weight"], d_best["pop_wt"])
policy = pd.DataFrame({"national cut-off": {"recall_people_top20": nat_overall["recall_people_top20"], "precision_top20": nat_overall["precision_top20"],
                                            **{k: v for k, v in fr.fairness_summary(tables[best]).items() if k in ("recall_gap", "worst_group_recall")}},
                       "separate urban/rural cut-offs (policy)": {**quota_overall, **{k: v for k, v in fr.fairness_summary(t_quota).items() if k in ("recall_gap", "worst_group_recall")}}})
display(policy)
area_cmp = pd.DataFrame({"national: selected": tables[best].query("dimension == 'area'").set_index("group")["selection_rate"],
                         "national: poor reached": tables[best].query("dimension == 'area'").set_index("group")["recall_people_top20"],
                         "separate: selected": t_quota.query("dimension == 'area'").set_index("group")["selection_rate"],
                         "separate: poor reached": t_quota.query("dimension == 'area'").set_index("group")["recall_people_top20"]})
area_cmp

In [ ]:
fig, axes = pl.figure("Poverty model mitigations: accuracy, recall balance and calibration",
                      "Development set, out-of-fold. PR-AUC with one fold standard error; fairness indices across the 7 group dimensions",
                      width=14, height=4.6, ncols=4)
order = list(res.index)
labels = {"baseline": "Baseline (LightGBM)", "1_interaction_features": "1 Interaction features", "2a_subgroup_balanced_weights": "2a Subgroup weights",
          "2b_class_balanced_weights": "2b Class weights", "3a_group_recalibration": "3a Group recalibration"}
for i, (ax, col, title, scale) in enumerate(zip(axes, ["pr_auc_mean", "recall_gap", "worst_group_recall", "mean_abs_calibration_gap"],
                                               ["PR-AUC (higher better)", "Recall gap, pp (lower better)", "Worst-group recall, % (higher better)", "Mean |calibration gap|, pp (lower better)"],
                                               [1, 100, 100, 100])):
    v = res.loc[order, col] * scale
    y = np.arange(len(order))[::-1]
    colors = [pl.SERIES[0] if o != best else pl.SERIES[1] for o in order]
    if col == "pr_auc_mean":
        ax.hlines(y, v - res.loc[order, "pr_auc_se"], v + res.loc[order, "pr_auc_se"], color=pl.BASELINE, linewidth=2)
    ax.scatter(v, y, c=colors, s=60, zorder=3)
    for yi, vi in zip(y, v):
        ax.text(vi, yi + 0.25, f"{vi:.3f}" if scale == 1 else f"{vi:.1f}", ha="center", fontsize=8, color=pl.INK["primary"])
    ax.set_yticks(y, [labels[o] for o in order] if i == 0 else [""] * len(order), fontsize=9)
    ax.set_title(title, fontsize=9.5, color=pl.INK["secondary"], loc="left")
    ax.margins(x=0.25)
path = pl.save(fig, FIG / "mitigation_comparison.png", top=0.74, bottom=0.12, left=0.14, wspace=0.12)
display(Image(str(path)))

## 3. Remaining gaps after mitigation

In [ ]:
d_best = dev.assign(score=scores[best].to_numpy())
best_tab = fr.group_table_ci(d_best, "score", "poor", B=200)
best_tab.to_csv(OUT / f"group_metrics_poverty_{best}.csv", index=False)
cmp = pd.DataFrame({"households": pov_tab["households"].to_numpy(),
                    "recall baseline": pov_tab["recall_people_top20"].to_numpy(), "recall chosen": best_tab["recall_people_top20"].to_numpy(),
                    "recall separate cut-offs": t_quota["recall_people_top20"].to_numpy(),
                    "error baseline": pov_tab["mean_error"].to_numpy(), "error chosen": best_tab["mean_error"].to_numpy(),
                    "error chosen CI low": best_tab["mean_error_ci_low"].to_numpy(), "error chosen CI high": best_tab["mean_error_ci_high"].to_numpy(),
                    "AUC baseline": pov_tab["roc_auc"].to_numpy(), "AUC chosen": best_tab["roc_auc"].to_numpy()},
                   index=pd.MultiIndex.from_frame(pov_tab[["dimension", "group"]]))
cmp.to_csv(OUT / "poverty_group_comparison.csv")
cmp

In [ ]:
t = cmp.reset_index()
t.index = [f"{d.replace('_', ' ')}: {g}" for d, g in zip(t["dimension"], t["group"])]
n = len(t)
h = 1.9 + 0.27 * n
fig, (a1, a2) = pl.figure(f"Poverty model: recall and calibration by group, baseline vs {labels.get(best, best)}",
                          "Development, out-of-fold. Grey = baseline, orange = chosen approach, green = chosen approach with separate urban/rural cut-offs (policy option)",
                          width=13, height=h, ncols=2)
y = np.arange(n)[::-1]
a1.scatter(100 * t["recall baseline"], y, color=pl.NEUTRAL, s=30, zorder=3)
a1.scatter(100 * t["recall chosen"], y, color=pl.SERIES[1], s=30, zorder=4)
a1.scatter(100 * t["recall separate cut-offs"], y, color=pl.SERIES[2], s=30, marker="D", zorder=4)
a1.set_yticks(y, [f"{i}  (n={hh:,})" for i, hh in zip(t.index, t["households"])], fontsize=8.5)
a1.set_title("Poor people selected at 20% budget, %", fontsize=9.5, color=pl.INK["secondary"], loc="left")
a2.axvline(0, color=pl.INK["muted"], linestyle="--", linewidth=1)
a2.scatter(100 * t["error baseline"], y, color=pl.NEUTRAL, s=30, zorder=3)
a2.hlines(y, 100 * t["error chosen CI low"], 100 * t["error chosen CI high"], color=pl.SERIES[1], alpha=0.4, linewidth=2)
a2.scatter(100 * t["error chosen"], y, color=pl.SERIES[1], s=30, zorder=4)
a2.set_yticks(y, [""] * n)
a2.set_title("Mean prediction error, pp (chosen approach with 95% interval)", fontsize=9.5, color=pl.INK["secondary"], loc="left")
for ax in (a1, a2):
    ax.set_ylim(-0.7, n - 0.3)
path = pl.save(fig, FIG / "poverty_recall_calibration_by_group.png", top=1 - 1.3 / h, bottom=0.5 / h, left=0.25, wspace=0.08)
display(Image(str(path)))

In [ ]:
res.to_csv(OUT / "poverty_mitigation_comparison.csv")
remaining = best_tab[best_tab["reliable"]].assign(abs_error=lambda x: x["mean_error"].abs()).sort_values("abs_error", ascending=False)
significant = best_tab[(best_tab["mean_error_ci_high"] < 0) | (best_tab["mean_error_ci_low"] > 0)]
print("Groups whose calibration interval still excludes zero (chosen approach):")
display(significant[["dimension", "group", "households", "observed_rate", "mean_predicted", "mean_error", "mean_error_ci_low", "mean_error_ci_high"]])
lowest = best_tab[best_tab["reliable"]].nsmallest(5, "recall_people_top20")[["dimension", "group", "observed_rate", "recall_people_top20", "recall_people_top20_ci_low", "recall_people_top20_ci_high"]]
print("Lowest recall at the national 20% cut-off (chosen approach):")
display(lowest)
choice = {"chosen_approach": best, "rule": "PR-AUC within 1 SE of best; best mean rank on recall gap, worst-group recall, calibration gap; ties to simpler",
          "summary": ranked.loc[best, ["pr_auc_mean", "pr_auc_se", "recall_gap", "worst_group_recall", "mean_abs_calibration_gap", "max_abs_calibration_gap"]].round(4).to_dict(),
          "baseline_summary": ranked.loc["baseline", ["pr_auc_mean", "pr_auc_se", "recall_gap", "worst_group_recall", "mean_abs_calibration_gap", "max_abs_calibration_gap"]].round(4).to_dict(),
          "policy_option": "separate urban/rural cut-offs (area_quota_inclusion); a policy decision, not part of the model",
          "policy_option_effect": policy.round(4).to_dict()}
(OUT / "mitigation_choice.json").write_text(json.dumps(choice, indent=2, default=float))
print(json.dumps(choice, indent=2, default=float))

## 4. Findings

1. **The formal rule keeps the baseline.** The pre-registered rule (`choose_approach`) ranks the unmitigated LightGBM first.
   - **Group recalibration (3a)** has the best PR-AUC (0.747 vs 0.744). It nearly removes the calibration gaps: the mean |gap| falls from 1.1 to 0.1 percentage points and the largest from 4.1 to 1.2.
   - **But it ranks second** because recall balance is slightly worse (recall gap 0.207 vs 0.194; worst-group recall 0.172 vs 0.185).
   - **Why the rank rule misleads here.** A rank average ignores the size of differences: a large calibration gain counts the same as two tiny recall losses that are within sampling noise.
   - **Next step.** Notebook 13 therefore carries group recalibration forward as a candidate, under a selection rule that uses standard-error tolerances.
2. **Interaction features (1)** change almost nothing: the trees already find these interactions.
3. **Subgroup-balanced weights (2a)** fail the PR-AUC gate and lower worst-group recall to 0.12.
4. **Class-balanced weights (2b)** distort calibration: predictions are 7.7 points too high on average.
5. **Separate urban/rural cut-offs (3b, policy option)** raise the share of urban poor reached from 58% to 62%, at almost no overall cost (66.5% to 66.3% of poor people reached nationally).
6. **Remaining gaps.** Fairness is not solved.
   - **Calibration.** The model under-predicts poverty in the Western (−4.1 pp) and Southern (−2.3 pp) provinces, for households with a disabled member (−2.9 pp) and for heads aged 60+ (−1.4 pp). It over-predicts in the Northern Province (+2.0 pp). The 95% intervals exclude zero.
   - **Recall.** At a national 20% budget, one-person households (19%) and 2–3 person households (42%) have far lower recall than larger households (66–74%). Kigali (53%), heads under 35 (54%) and non-farm self-employed households (54%) are also low.
   - **Partly a property of the cut-off.** These groups have low poverty rates and are poor per adult equivalent. Within-group ranking quality stays high (AUC 0.85–0.94).
7. **VUP coverage model.** Group calibration gaps are within ±1.7 points, except for households of 8 or more members (+3.7, interval −1.0 to +8.3, 15 recipients). Fairness indices are imprecise, with 20–100 recipients per group, so they were not used to change that model here.